# Phase 2 — Train the REAL spectral LiteFNO (Gray-Scott)

This notebook implements the **actual** LiteFNO architecture the paper describes
— a **CP-factorized spectral FNO** (via `neuraloperator`) — and trains it on
Gray-Scott. The repo's existing `LiteFNO` class is a CNN and is *not* used here.

Output (for Phase 3):
- `litefno_real_best.pt` / `litefno_real_last.pt` — checkpoints
- `gray_scott_litefno_real.jsonl` — per-epoch metrics

**After it finishes**, save the notebook so `/kaggle/working/extensions/` becomes
the notebook output, then in Phase 3 add this notebook's output as an input
dataset so Phase 3 can load `litefno_real_best.pt`.

**Setup:** Kaggle Settings -> Internet ON, Accelerator = GPU.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/AIscend-Research/litefno-repro"
REPO_DIR = "litefno-repro"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "neuraloperator", "the_well", "thop"], check=False)
print("cwd:", os.getcwd())

In [ ]:
import json, time, math
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import DataLoader

from litefno.data import DatasetConfig, H5SequenceDataset
from litefno.train import flatten_time
from litefno.metrics import rmse, vrmse
from litefno.download import download_dataset
from litefno.preprocess import preprocess_well_split

torch.manual_seed(1337); np.random.seed(1337)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":   # P100 (sm_60) is incompatible with new torch; verify it works
    try:
        _ = (torch.zeros(1, device=DEVICE) + 1).item()
    except Exception as e:
        print("CUDA unusable (use T4, not P100):", e); DEVICE = torch.device("cpu")
print("device:", DEVICE)
if DEVICE.type != "cuda":
    print("WARNING: not on GPU -> use Settings > Accelerator > GPU T4 x2 (NOT P100), "
          "otherwise training is far too slow.")

OUT = Path("/kaggle/working/extensions") if Path("/kaggle/working").exists() else Path("extensions_out")
OUT.mkdir(parents=True, exist_ok=True)

# --- Gray-Scott config (matches the repo's GS preprocessing) ---
DATASET="gray_scott_reaction_diffusion"; KEY="data"; FIELDS=2
DOWNSAMPLE=4; MAX_TRAJ=1000; MAX_STEPS=60; SEED=0
RAW=Path("/kaggle/temp/gs_raw") if Path("/kaggle").exists() else Path("data/raw/gs_ext"); PROC=Path("data/processed/gs_ext")
# If you uploaded the preprocessed train/valid/test.h5 as a Kaggle Dataset, set this
# to its mount path; Phase 2 will use it and SKIP the 19.5 GB download entirely.
PROC_INPUT = Path("/kaggle/input/gs-processed")

# --- training protocol (matched to the repo CNN runs for a fair comparison) ---
WIDTH=64; LAYERS=8; RANK=0.5; FACT="cp"
EPOCHS=200; BATCH=64; LR=1e-3; LR_STEP=100; LR_GAMMA=0.5
# (paper-faithful would be EPOCHS=500 + a transduction stage; see note at end)

## Download + preprocess Gray-Scott (train / valid / test)

In [ ]:
if (PROC_INPUT / "train.h5").exists():
    PROC = PROC_INPUT
    print("Using PRE-UPLOADED processed data:", PROC, "(skipping download)")
else:
    print("No pre-uploaded data found; downloading raw from The Well (large).")
    RAW.mkdir(parents=True, exist_ok=True)
    for split in ["train", "valid", "test"]:
        out_h5 = PROC / f"{split}.h5"
        if out_h5.exists():
            continue
        download_dataset(DATASET, split, RAW)
        PROC.mkdir(parents=True, exist_ok=True)
        preprocess_well_split(RAW, out_h5, DATASET, split, KEY, DOWNSAMPLE, MAX_TRAJ, MAX_STEPS, random_seed=SEED)
        import shutil  # free disk immediately: raw split is ~19.5 GB, processed is tiny
        raw_split = RAW / "datasets" / DATASET / "data" / split
        if raw_split.exists():
            shutil.rmtree(raw_split); print(f"  freed raw '{split}'")

def make_loader(split, bs, shuffle):
    cfg = DatasetConfig(path=PROC / f"{split}.h5", dataset_key=KEY,
                        input_steps=1, output_steps=1, stride=1, cache="memory")
    return DataLoader(H5SequenceDataset(cfg), batch_size=bs, shuffle=shuffle)

train_loader = make_loader("train", BATCH, True)
valid_loader = make_loader("valid", BATCH, False)
test_loader  = make_loader("test",  BATCH, False)

import h5py
with h5py.File(PROC / "train.h5", "r") as f:
    H, W = f[KEY].shape[2], f[KEY].shape[3]
MODES = min(16, H // 2)
print(f"resolution {H}x{W}  modes={MODES}  train batches={len(train_loader)}")

## Build the real spectral LiteFNO

In [ ]:
def build_real_litefno(in_ch, out_ch, modes, width=64, layers=8, rank=0.5, factorization="cp"):
    """Real spectral LiteFNO: a CP/Tucker-factorized FNO (neuraloperator).

    Tries the requested factorization, then tucker, then a dense FNO, so the
    notebook still produces a genuine *spectral* operator even if a particular
    factorization API is unavailable. Returns (model, kind_used).
    """
    from neuralop.models import FNO
    base = dict(n_modes=(modes, modes), hidden_channels=width,
                in_channels=in_ch, out_channels=out_ch, n_layers=layers)
    fac = None if factorization in (None, "dense") else factorization
    attempts = [(fac, rank), ("tucker", rank), (None, None)]
    last = None
    for f, r in attempts:
        try:
            if f is None:
                return FNO(**base), "dense"
            return FNO(**base, factorization=f, rank=r), f
        except Exception as e:  # noqa: BLE001
            last = e
            print(f"  [build] factorization={f} failed: {e}")
    raise RuntimeError(f"could not construct FNO: {last}")

model, KIND = build_real_litefno(FIELDS, FIELDS, MODES, WIDTH, LAYERS, RANK, FACT)
model = model.to(DEVICE)
PARAMS = sum(p.numel() for p in model.parameters())
print(f"Built real LiteFNO  factorization={KIND}  params={PARAMS:,}")
BUILD = {"modes": MODES, "width": WIDTH, "layers": LAYERS, "rank": RANK,
         "factorization": KIND, "in_ch": FIELDS, "out_ch": FIELDS}

## Helpers: train + eval one epoch

In [ ]:
def to_xy(batch):
    xb, yb = batch                     # (B,1,H,W,2)
    return flatten_time(xb).to(DEVICE), flatten_time(yb).to(DEVICE)  # (B,2,H,W)

scaler = torch.amp.GradScaler("cuda") if DEVICE.type == "cuda" else None

def train_epoch(opt):
    model.train(); tot = 0.0
    for batch in train_loader:
        x, y = to_xy(batch)
        opt.zero_grad(set_to_none=True)
        if scaler is not None:
            with torch.amp.autocast("cuda"):
                loss = torch.nn.functional.mse_loss(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        else:
            loss = torch.nn.functional.mse_loss(model(x), y)
            loss.backward(); opt.step()
        tot += loss.item()
    return tot / max(1, len(train_loader))

@torch.no_grad()
def eval_loader(loader):
    model.eval(); rs = []; vs = []
    for batch in loader:
        x, y = to_xy(batch)
        p = model(x)
        rs.append(rmse(p, y).item()); vs.append(vrmse(p, y).item())
    return float(np.mean(rs)), float(np.mean(vs))

## Timing check (3 epochs) — estimate full run before committing

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=LR)
t0 = time.time()
for _ in range(3):
    train_epoch(opt)
per_epoch = (time.time() - t0) / 3
print(f"~{per_epoch:.1f} s/epoch  ->  full {EPOCHS} epochs ~= {per_epoch*EPOCHS/3600:.1f} GPU-hours")
print("If that's too long, lower EPOCHS or use a smaller MODES/WIDTH and re-run from the build cell.")

## Full training (fresh model + optimizer)

In [ ]:
# rebuild fresh so the 3 probe epochs don't count
model, KIND = build_real_litefno(FIELDS, FIELDS, MODES, WIDTH, LAYERS, RANK, FACT)
model = model.to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.StepLR(opt, step_size=LR_STEP, gamma=LR_GAMMA)

log_path = OUT / "gray_scott_litefno_real.jsonl"
best_v = float("inf")
with open(log_path, "w") as logf:
    for epoch in range(EPOCHS):
        loss = train_epoch(opt)
        tr_r, tr_v = eval_loader(train_loader)
        va_r, va_v = eval_loader(valid_loader)
        rec = {"step": epoch, "loss": loss, "params": PARAMS,
               "train_rmse": tr_r, "train_vrmse": tr_v,
               "valid_rmse": va_r, "valid_vrmse": va_v}
        logf.write(json.dumps(rec) + "\n"); logf.flush()
        if va_v < best_v:
            best_v = va_v
            torch.save({"epoch": epoch, "model_state": model.state_dict(), "build": BUILD},
                       OUT / "litefno_real_best.pt")
        if epoch % 10 == 0 or epoch == EPOCHS - 1:
            print(f"epoch {epoch:>3}  loss={loss:.4e}  valid_vrmse={va_v:.5f}  (best={best_v:.5f})")
        sched.step()
torch.save({"epoch": EPOCHS, "model_state": model.state_dict(), "build": BUILD},
           OUT / "litefno_real_last.pt")
print("saved checkpoints to", OUT)

## Test evaluation

In [ ]:
ck = torch.load(OUT / "litefno_real_best.pt", map_location=DEVICE)
model.load_state_dict(ck["model_state"]); model.to(DEVICE)
te_r, te_v = eval_loader(test_loader)
print(f"REAL LiteFNO  test RMSE={te_r:.6f}  test VRMSE={te_v:.6f}  params={PARAMS:,}  factorization={KIND}")
with open(log_path, "a") as logf:
    logf.write(json.dumps({"step": EPOCHS, "test_rmse": te_r, "test_vrmse": te_v}) + "\n")
print("Compare against: repo CNN GS test_vrmse=0.0227 ; paper LiteFNO GS one-step VRMSE=0.0098")

## Handoff to Phase 3 + notes

- This notebook's `/kaggle/working/extensions/litefno_real_best.pt` is the real
  LiteFNO arm. **Save the notebook**, then in Phase 3 use *Add Input -> Notebook
  Output* to mount it; set `REAL_CKPT` there to its path.
- **Deviations from the paper (document these):** matched-protocol training
  (EPOCHS=200, MSE loss, batch 64, single-step) for a fair head-to-head with the
  CNN arm, rather than the paper's 500 epochs + relative-L2 + transduction
  fine-tuning. The CP `rank` here is `neuraloperator`'s fractional rank, not the
  paper's integer rank {32,48}. For a paper-faithful run, raise EPOCHS to 500 and
  add a transduction (spatio-temporal, S=3) fine-tuning stage.